# Python Mastery — Module 10
# Natural Language Processing and Transformers

A practical beginner-to-advanced module covering text preprocessing, tokenization, word embeddings, attention, transformer blocks, pretrained language models, fine-tuning, and evaluation.

## Learning outcomes
- Explain common NLP tasks and text preprocessing trade-offs.
- Tokenize text and understand token IDs, vocabulary, padding, truncation, and masks.
- Compare bag-of-words, TF-IDF, static word embeddings, and contextual embeddings.
- Understand attention and the Transformer encoder/decoder at a conceptual level.
- Use a pretrained language model when the library and model weights are available.
- Fine-tune a small pretrained model responsibly, or understand the complete workflow when compute/network access is unavailable.
- Evaluate text models with task-appropriate metrics and error analysis.
- Avoid leakage, biased data, and unsupported claims from small evaluation sets.

**Important:** Examples use public toy text or synthetic data. They are for learning, not evidence of suitability for consequential decisions.

## 1. What is NLP?

Natural Language Processing (NLP) builds systems that work with human language. Common tasks include:
- text classification (spam, sentiment, topic);
- named-entity recognition (people, places, organizations);
- information extraction and question answering;
- summarization and translation;
- semantic search and retrieval;
- conversational assistants and generation.

A typical pipeline is: define the task → collect and inspect text → split data correctly → represent text → train or adapt a model → evaluate → analyze errors → deploy and monitor.

## 2. Text preprocessing: useful, but not one-size-fits-all

Common operations include lowercasing, whitespace cleanup, Unicode normalization, HTML cleanup, language detection, and removing malformed records.

Traditional ML pipelines often use stop-word filtering and stemming/lemmatization. Modern pretrained transformer models generally expect text close to their training-time conventions; aggressive cleaning can remove negation, punctuation, casing, or formatting that carries meaning.

Examples:
- “not good” must not become “good”;
- punctuation can matter for intent and sentiment;
- URLs and hashtags may contain useful information;
- personally identifying information should be handled under the data governance policy.

In [ ]:
import re
import unicodedata
import pandas as pd
import numpy as np

texts = [
    "The service was NOT good!!!",
    "Loved the new dashboard 😊",
    "Payment failed after 3 attempts.",
    "The claim was approved.",
    "The claim was not approved.",
]

def basic_clean(text):
    # Conservative cleaning: normalize Unicode and whitespace, retain punctuation.
    text = unicodedata.normalize("NFKC", str(text))
    text = re.sub(r"\s+", " ", text).strip()
    return text

pd.DataFrame({
    "original": texts,
    "cleaned": [basic_clean(t) for t in texts],
})

## 3. Tokenization: turning text into units

A tokenizer splits text into tokens and maps them to integer IDs. Tokens may be words, subwords, characters, or byte-level units.

Subword tokenization helps represent rare words by composing them from smaller pieces. The model's vocabulary and tokenizer are normally a matched pair: using the wrong tokenizer can make the IDs meaningless to the model.

Important concepts:
- **Vocabulary:** known token-to-ID mapping.
- **Special tokens:** markers such as beginning/end of sequence or classification separators.
- **Padding:** make sequences in a batch a common length.
- **Truncation:** shorten sequences to a maximum length.
- **Attention mask:** indicates which positions are real tokens versus padding for many models.

In [ ]:
# A tiny illustrative word tokenizer; this is NOT a modern subword tokenizer.
def word_tokenize_demo(text):
    return re.findall(r"\w+|[^\w\s]", text.lower(), flags=re.UNICODE)

for text in texts[:3]:
    print(text, "->", word_tokenize_demo(text))

## 4. Bag-of-words and TF-IDF baselines

Before using transformers, build a simple baseline. Bag-of-words counts tokens; TF-IDF weights terms by how informative they are across documents.

Advantages:
- fast and often surprisingly competitive for short text classification;
- interpretable through term weights;
- low compute requirements.

Limitations:
- word order and long-range context are mostly lost;
- synonyms may not share a representation;
- vocabulary can become large and sparse.

The following example uses a small synthetic sentiment dataset. Treat the scores as a demonstration only.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

dataset = [
    ("The service was excellent and fast", 1),
    ("Very helpful support team", 1),
    ("The app is easy to use", 1),
    ("I am happy with the result", 1),
    ("Quick response and great experience", 1),
    ("The product works perfectly", 1),
    ("The service was terrible and slow", 0),
    ("Support was unhelpful", 0),
    ("The app crashes every time", 0),
    ("I am unhappy with the result", 0),
    ("Long wait and poor experience", 0),
    ("The product does not work", 0),
    ("Really good and reliable", 1),
    ("Not good and not reliable", 0),
    ("Excellent response from the team", 1),
    ("Awful response from the team", 0),
    ("The claim was approved", 1),
    ("The claim was not approved", 0),
    ("Payment succeeded", 1),
    ("Payment failed", 0),
    ("Clear explanation and friendly staff", 1),
    ("Confusing explanation and rude staff", 0),
    ("The report is accurate", 1),
    ("The report is incorrect", 0),
]
df = pd.DataFrame(dataset, columns=["text", "label"])
X_train_text, X_test_text, y_train_text, y_test_text = train_test_split(
    df["text"], df["label"], test_size=0.25, stratify=df["label"], random_state=42
)
tfidf_model = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True)),
    ("classifier", LogisticRegression(max_iter=2000, random_state=42)),
])
tfidf_model.fit(X_train_text, y_train_text)
tfidf_pred = tfidf_model.predict(X_test_text)
print("Accuracy:", round(accuracy_score(y_test_text, tfidf_pred), 3))
print("Precision:", round(precision_score(y_test_text, tfidf_pred, zero_division=0), 3))
print("Recall:", round(recall_score(y_test_text, tfidf_pred, zero_division=0), 3))
print("F1:", round(f1_score(y_test_text, tfidf_pred, zero_division=0), 3))
print("\nTest examples and predictions:")
pd.DataFrame({"text": X_test_text, "actual": y_test_text, "predicted": tfidf_pred})

## 5. Word embeddings

An embedding maps a discrete token or item to a dense vector of numbers. Similar words may have vectors that are close under a chosen similarity measure, depending on how the embeddings were trained.

- **Static embeddings** (e.g., Word2Vec, GloVe): a word typically has one vector regardless of context.
- **Contextual embeddings** (e.g., transformer hidden states): representation changes with surrounding words.

For example, “bank” in “river bank” and “bank account” can have different contextual representations. Embeddings are learned representations, not guaranteed definitions or unbiased semantic truths.

In [ ]:
# A tiny conceptual embedding table. Values are illustrative, not trained embeddings.
embedding_table = {
    "excellent": np.array([0.9, 0.8, 0.1]),
    "great": np.array([0.85, 0.75, 0.15]),
    "terrible": np.array([-0.8, -0.7, 0.2]),
    "slow": np.array([-0.2, -0.6, 0.9]),
}
def cosine_similarity(a, b):
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom else 0.0

print("excellent vs great:", round(cosine_similarity(embedding_table["excellent"], embedding_table["great"]), 3))
print("excellent vs terrible:", round(cosine_similarity(embedding_table["excellent"], embedding_table["terrible"]), 3))

## 6. Attention: the central Transformer idea

Attention lets a token build a representation by weighting information from other tokens. In self-attention, queries, keys, and values are derived from the same input sequence.

Scaled dot-product attention is commonly written as:

\[
\mathrm{Attention}(Q,K,V)=\mathrm{softmax}\left(\frac{QK^T}{\sqrt{d_k}}+M\right)V
\]

- **Query (Q):** what this position is looking for.
- **Key (K):** what each position offers for matching.
- **Value (V):** information combined using attention weights.
- **Mask (M):** can block padding or future positions, depending on the task.
- **Scaling:** prevents dot products from growing too large as key dimensions increase.

Attention weights show model computation, but they are not automatically faithful explanations of why a prediction was made.

In [ ]:
# A tiny self-attention calculation with NumPy (one head, toy numbers).
tokens = np.array([
    [1.0, 0.0],  # token 1 representation
    [0.0, 1.0],  # token 2 representation
    [1.0, 1.0],  # token 3 representation
])
Q = tokens
K = tokens
V = tokens
scores = (Q @ K.T) / np.sqrt(K.shape[1])
scores_shifted = scores - scores.max(axis=1, keepdims=True)
weights = np.exp(scores_shifted)
weights = weights / weights.sum(axis=1, keepdims=True)
context = weights @ V
print("Attention weights (each row sums to 1):\n", np.round(weights, 3))
print("Context representations:\n", np.round(context, 3))

## 7. Multi-head attention and positional information

**Multi-head attention** runs several attention operations in parallel. Different heads can learn different relationships, although their functions are not always neatly interpretable.

Self-attention alone does not inherently encode token order. Transformers therefore add positional information, using learned position embeddings, sinusoidal encodings, or other position schemes.

A Transformer block commonly includes:
1. multi-head attention;
2. residual connection and normalization;
3. feed-forward network applied at each position;
4. another residual connection and normalization.

Exact order and normalization variants differ across model families.

## 8. Encoder, decoder, and model families

- **Encoder-only models** (e.g., BERT-like): contextual representations useful for classification, tagging, and retrieval embeddings.
- **Decoder-only models** (many generative LLMs): predict the next token using causal attention.
- **Encoder-decoder models** (e.g., T5-like): encode input text and generate an output sequence; often used for translation and summarization.

A decoder-only language model is trained to predict the next token. At generation time, it repeatedly chooses or samples a next token and appends it to the context. Output quality depends on the model, prompt, decoding settings, and constraints.

## 9. Tokenization and a pretrained Transformer with Hugging Face

The `transformers` library is widely used to load pretrained tokenizers and models. Model weights may need to be downloaded, so this section is optional in offline environments.

Before using a model:
- check license and intended use;
- inspect model card and training data notes;
- use the matching tokenizer;
- set truncation and maximum length deliberately;
- avoid sending sensitive text to a remote service without authorization;
- record model revision and package versions for reproducibility.

In [ ]:
try:
    import transformers
    print("Transformers version:", transformers.__version__)
except ImportError:
    transformers = None
    print("The transformers package is not installed. You can install it separately to run the pretrained-model section.")

In [ ]:
# Optional: load a small pretrained sentiment classifier.
# This requires internet access the first time, or a cached local model.
if transformers is not None:
    try:
        from transformers import pipeline
        sentiment_pipe = pipeline("sentiment-analysis", model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")
        print(sentiment_pipe([
            "The explanation was clear and useful.",
            "The service was slow and frustrating."
        ]))
    except Exception as exc:
        print("Could not load the pretrained model:", type(exc).__name__, str(exc)[:200])
        print("If offline, skip this cell or load a model already cached locally.")
else:
    print("Skipped: transformers is not installed.")

## 10. Fine-tuning a pretrained model

Fine-tuning continues training a pretrained model on labeled examples for a specific task. It can outperform a frozen model when enough representative labeled data exist, but can overfit or lose useful general behavior when data are small or the learning rate is too high.

Typical workflow:
1. choose a model suited to the task and language;
2. split data into training, validation, and test sets;
3. tokenize using the model's tokenizer;
4. create batches with padding/truncation;
5. add or adapt the task head;
6. train with a small learning rate;
7. monitor validation metrics and early-stop;
8. evaluate once on test data;
9. document limitations and version details.

**Parameter-efficient fine-tuning (PEFT)** methods such as LoRA train a smaller set of added parameters rather than updating every model weight. This can reduce memory and compute requirements, but still requires careful evaluation.

### Conceptual fine-tuning configuration

The next cell illustrates a configuration rather than running a potentially expensive training job. The exact APIs can vary by Transformers version; consult the installed version's documentation before running a full training pipeline.

In [ ]:
fine_tuning_plan = pd.DataFrame([
    ("Base model", "A pretrained encoder or task-appropriate model"),
    ("Input", "Tokenized text with attention masks"),
    ("Training objective", "Task-specific cross-entropy or another suitable loss"),
    ("Learning rate", "Start small, often around 1e-5 to 5e-5 for full fine-tuning; validate"),
    ("Batch size", "Choose based on sequence length and available memory"),
    ("Validation", "Use for checkpoint selection and early stopping"),
    ("Test", "Keep untouched until final evaluation"),
    ("Reproducibility", "Record model revision, tokenizer, seed, data split, and package versions"),
], columns=["component", "decision / explanation"])
fine_tuning_plan

## 11. Evaluation metrics for NLP

Select metrics according to the task and error costs.

### Classification
- Accuracy: fraction of predictions correct.
- Precision: among predicted positives, fraction truly positive.
- Recall: among actual positives, fraction found.
- F1: harmonic mean of precision and recall.
- Macro F1: compute F1 per class and average equally; useful when each class matters.
- Confusion matrix: reveals which classes are confused.

### Retrieval and ranking
- Precision@k / Recall@k: relevance among top-k results or recovered relevant items.
- Mean reciprocal rank (MRR): rewards placing the first relevant result high.
- nDCG: considers ranking positions and graded relevance.

### Generation
- Exact match / token F1 for some question-answering tasks.
- BLEU / ROUGE for specific translation/summarization comparisons.
- Human or task-specific evaluation for factuality, relevance, safety, and usefulness.

No single metric captures all qualities of a language model. Automatic scores may correlate imperfectly with human judgments.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
ConfusionMatrixDisplay.from_predictions(y_test_text, tfidf_pred)
plt.title("TF-IDF baseline confusion matrix")
plt.show()

## 12. Error analysis and robust evaluation

After calculating a score, inspect individual errors:
- false positives and false negatives;
- negation, sarcasm, ambiguous language, and domain-specific vocabulary;
- class imbalance and low-frequency labels;
- performance by language, source, time, or other relevant groups;
- examples that are duplicates or near-duplicates across splits;
- label quality and annotator disagreement.

Use a held-out test set that reflects the intended deployment distribution. If multiple examples come from the same person, customer, conversation, or source, split by group when needed. For time-sensitive tasks, consider temporal validation.

In [ ]:
error_table = pd.DataFrame({
    "text": X_test_text.to_numpy(),
    "actual": y_test_text.to_numpy(),
    "predicted": tfidf_pred,
})
errors = error_table[error_table["actual"] != error_table["predicted"]]
print(f"Misclassified examples: {len(errors)} of {len(error_table)}")
errors

## 13. Embedding similarity and semantic search

Text embeddings can support semantic search by mapping queries and documents to vectors. Similarity is often measured with cosine similarity or dot product, depending on the embedding model and retrieval setup.

A simple retrieval workflow:
1. split documents into useful chunks;
2. embed each chunk with a model intended for retrieval;
3. store vectors with source metadata;
4. embed the query;
5. retrieve top-k similar chunks;
6. optionally rerank candidates;
7. evaluate retrieval relevance separately from answer generation.

Generic language-model hidden states are not always good retrieval embeddings. Prefer models trained or evaluated for semantic search, and measure retrieval quality on representative queries.

In [ ]:
# A tiny semantic-search demonstration using TF-IDF vectors.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

documents = [
    "How to reset a password and regain account access",
    "Guidance for submitting a medical claim",
    "Troubleshooting a failed payment",
    "Steps to update profile information",
]
query = "My payment did not go through"
vectorizer = TfidfVectorizer(ngram_range=(1, 2))
doc_vectors = vectorizer.fit_transform(documents)
query_vector = vectorizer.transform([query])
similarities = cosine_similarity(query_vector, doc_vectors).ravel()
pd.DataFrame({"document": documents, "similarity": similarities}).sort_values("similarity", ascending=False)

## 14. Practical NLP model-selection guide

- **Small labeled dataset, strong baseline needed:** TF-IDF + logistic regression or linear SVM.
- **Short text classification with contextual ambiguity:** try a pretrained encoder and fine-tune if resources allow.
- **Named-entity recognition:** use a token-classification model with carefully aligned token labels.
- **Semantic search:** use an embedding model trained for retrieval and evaluate Recall@k / nDCG.
- **Summarization or generation:** use a suitable encoder-decoder or generative model; evaluate factuality and task-specific quality.
- **Low latency or limited hardware:** consider smaller models, quantization, distillation, or simpler baselines.
- **Sensitive data:** consider approved local processing, access controls, retention policies, and privacy review.

Always compare with a simple baseline and select based on the intended task, not model popularity.

## 15. Common mistakes

1. Removing negation or meaningful punctuation during cleaning.
2. Using a tokenizer that does not match the pretrained model.
3. Truncating long texts without understanding what information is lost.
4. Evaluating only on a tiny or unrepresentative test set.
5. Tuning on test results repeatedly.
6. Treating attention weights as complete explanations.
7. Assuming embeddings are unbiased or always semantically correct.
8. Fine-tuning with too high a learning rate or too few representative examples.
9. Using accuracy alone for imbalanced classification.
10. Confusing retrieval quality with answer-generation quality.
11. Ignoring model license, provenance, privacy, or deployment cost.

## 16. Exercises

Try to answer before reading the suggested answers.

1. Why can aggressive text cleaning harm a transformer model?
2. What is the difference between a token and a token ID?
3. Why are subword tokenizers useful?
4. What is the role of an attention mask?
5. Explain query, key, and value in self-attention.
6. How does a decoder-only language model generate text?
7. What is the difference between an encoder-only and encoder-decoder model?
8. Why should the tokenizer match the pretrained model?
9. When might TF-IDF outperform a transformer baseline?
10. Why is model evaluation more than one metric?

### Suggested answers

1. It may remove casing, punctuation, negation, or formatting the model relies on.
2. A token is a text unit; a token ID is its integer index in a particular vocabulary.
3. They can represent rare or unseen words as smaller known pieces.
4. It helps the model distinguish actual tokens from padding, or blocks positions where attention is disallowed.
5. Queries represent what a token seeks, keys represent what tokens offer for matching, and values provide information combined using attention weights.
6. It repeatedly predicts a next token based on the preceding context and appends it.
7. Encoder-only models build contextual representations; encoder-decoder models encode input and generate an output sequence through a decoder.
8. Token IDs are meaningful only relative to the model's vocabulary and tokenization scheme.
9. When data are small, task patterns are lexical, compute is limited, or a simple linear boundary works well.
10. Different metrics measure different failure modes; task costs, class balance, ranking, factuality, and user needs differ.

## 17. Capstone — text classification or semantic search

Choose one path.

### Path A: text classification
1. Define classes and intended use.
2. Inspect class balance, duplicates, language, and label quality.
3. Create leakage-safe train/validation/test splits.
4. Train a TF-IDF baseline.
5. Evaluate per-class metrics and inspect errors.
6. Optionally compare a pretrained Transformer classifier.
7. Evaluate on a held-out test set and document limitations.

### Path B: semantic search
1. Gather a small set of documents and realistic user queries.
2. Chunk documents and preserve source metadata.
3. Build a TF-IDF baseline.
4. Optionally embed with a retrieval-oriented model.
5. Label relevant documents for queries.
6. Compare Recall@k, MRR, or nDCG.
7. Inspect misses and document retrieval limitations.

### Model card / system card
- Intended use and out-of-scope use
- Data provenance, language coverage, and licensing
- Preprocessing/tokenizer/model version
- Split design and evaluation metrics
- Error analysis and subgroup checks
- Privacy and security controls
- Known limitations and monitoring plan

## 18. Readiness checklist

- [ ] Explain NLP tasks and conservative text preprocessing.
- [ ] Explain tokens, IDs, vocabulary, padding, truncation, and masks.
- [ ] Build a TF-IDF baseline and evaluate it.
- [ ] Explain static versus contextual embeddings.
- [ ] Explain queries, keys, values, and attention masks.
- [ ] Distinguish encoder-only, decoder-only, and encoder-decoder models.
- [ ] Explain pretrained-model inference and fine-tuning.
- [ ] Select metrics suitable for classification, retrieval, or generation.
- [ ] Inspect errors and use leakage-safe splits.
- [ ] Document privacy, licensing, limitations, and reproducibility.

### Next module
**Module 11 — Applied LLM Engineering and RAG:** prompting, structured outputs, embeddings, chunking, vector retrieval, reranking, grounding, evaluation, guardrails, and a practical retrieval-augmented generation application.